# NB_hospital_daily_summary
Builds `adb_caresync.gold.hospital_daily_summary` per STTM *Silver to Gold v1.0* (sheet `hospital_daily_summary`).

* **Grain:** one row per hospital per calendar day (days with at least one lab test)
* **Driving table:** `adb_caresync.silver.lab_results` grouped by `hospital_id`, `test_date`
* **Joins (LEFT):** `hospitals` on `hospital_id`; `calendar` on `date_key = CAST(date_format(test_date, 'yyyyMMdd') AS INT)`
* **Window:** `LAG(total_tests)` partitioned by `hospital_id`, ordered by `summary_date` (previous row = previous day *with tests* for that hospital)
* **Filters:** none
* **Load:** FULL rebuild via `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day of lab-test activity with day-over-day trend (STTM Silver to Gold v1.0)'
AS
WITH daily AS (
  -- Lab results aggregated to hospital x day grain
  SELECT
    hospital_id,
    test_date                             AS summary_date,
    COUNT(*)                              AS total_tests,
    COUNT(DISTINCT patient_id)            AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)             AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    *,
    LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) AS prev_day_total_tests
  FROM daily
)
SELECT
  d.hospital_id,
  h.hospital_name,
  d.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(d.total_tests AS INT)                                                             AS total_tests,
  CAST(d.unique_patients_tested AS INT)                                                  AS unique_patients_tested,
  CAST(d.active_doctor_count AS INT)                                                     AS active_doctor_count,
  CAST(d.abnormal_test_count AS INT)                                                     AS abnormal_test_count,
  CAST(d.critical_test_count AS INT)                                                     AS critical_test_count,
  CAST(d.abnormal_test_count * 100.0 / NULLIF(d.total_tests, 0) AS DECIMAL(5,2))         AS abnormal_rate_pct,
  CAST(d.critical_test_count * 100.0 / NULLIF(d.total_tests, 0) AS DECIMAL(5,2))         AS critical_rate_pct,
  CAST(d.prev_day_total_tests AS INT)                                                    AS prev_day_total_tests,   -- NULL on a hospital's first day
  CAST((d.total_tests - d.prev_day_total_tests) * 100.0
       / NULLIF(d.prev_day_total_tests, 0) AS DECIMAL(6,2))                              AS test_volume_change_pct  -- NULL if prev is NULL/0
FROM with_lag d
LEFT JOIN adb_caresync.silver.hospitals h
  ON d.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(d.summary_date, 'yyyyMMdd') AS INT);

In [0]:
source_table = "adb_caresync.silver.lab_results"
target_table = "adb_caresync.gold.hospital_daily_summary"

record_count = spark.table(source_table).count()
target_count = spark.table(target_table).count()
print(f"Step 6: Load finished. Source records processed: {record_count}, total records in {target_table}: {target_count}")
dbutils.notebook.exit(str(record_count))